# Criação da Dimensão Produto


In [0]:
df_dim_produto = spark.sql("""
                          SELECT
                              row_number() over(ORDER BY produto) AS id_produto, produto
                          FROM (
                            SELECT DISTINCT produto
                            FROM projeto_combustiveis.silver.historico_mensal_brasil
                          )""")

In [0]:
df_dim_produto.write.mode("overwrite").saveAsTable("projeto_combustiveis.gold.dim_produto")

In [0]:
display(df_dim_produto)

# Criando Dimensão Localidade

In [0]:
df_dim_localidade = spark.sql("""
    WITH todas_cidades AS (
        -- 1. Pega dados do histórico
        SELECT regiao, estado, municipio
        FROM projeto_combustiveis.silver.historico_mensal_municipios
        WHERE municipio IS NOT NULL
        
        UNION ALL
        
        -- 2. Pega dados semanais
        SELECT CAST(NULL AS STRING) AS regiao, estado, municipio
        FROM projeto_combustiveis.silver.monitor_semanal_municipios
        WHERE municipio IS NOT NULL
    ),
    
    -- 3. Cria um Dicionário automático de qual Estado pertence a qual Região
    mapa_regioes AS (
        SELECT estado, MAX(regiao) AS regiao_correta
        FROM todas_cidades
        WHERE regiao IS NOT NULL
        GROUP BY estado
    )
    
    -- 4. Junta tudo, remove duplicatas e gera o ID
    SELECT 
        ROW_NUMBER() OVER(ORDER BY t.estado, t.municipio) AS id_localidade,
        m.regiao_correta AS regiao,
        t.estado,
        t.municipio
    FROM todas_cidades t
    LEFT JOIN mapa_regioes m ON t.estado = m.estado
    GROUP BY m.regiao_correta, t.estado, t.municipio
""")

df_dim_localidade.write.mode("overwrite").saveAsTable("projeto_combustiveis.gold.dim_localidade")
display(df_dim_localidade)

# Criando Dimensão Tempo

In [0]:
df_dim_tempo = spark.sql("""
    WITH explode_dates AS (
        SELECT explode(sequence(to_date('2013-01-01'), to_date('2030-12-31'), interval 1 day)) AS data_referencia
    )
    SELECT 
        CAST(date_format(data_referencia, 'yyyyMMdd') AS INT) AS id_tempo,
        data_referencia,
        year(data_referencia) AS ano,
        month(data_referencia) AS mes,
        day(data_referencia) AS dia,
        quarter(data_referencia) AS trimestre,
        CASE 
            WHEN month(data_referencia) <= 6 THEN 1 
            ELSE 2 
        END AS semestre,
        date_format(data_referencia, 'MMMM') AS nome_mes,
        dayofweek(data_referencia) AS dia_semana
    FROM explode_dates
""")

In [0]:
df_dim_tempo.write.mode("overwrite").saveAsTable("projeto_combustiveis.gold.dim_tempo")
display(df_dim_tempo)